# Panoramic X-ray to 3D scan: Patient 1 (Kaggle)

This notebook does not train anything. It takes about 3 minutes.
It loads the trained model (`best.pt`) from the output of the notebook `xray-to-3d-training` and runs it
on Patient 1's real panoramic X-ray (private dataset `patient1-xray`).
The model's outputs are saved in `predictions/patient1/` (see the "Output" tab).
The comparison with Patient 1's real 3D scan is done on the Mac, because the real 3D scan must not leave the Mac.

In [ ]:
import torch
assert torch.cuda.is_available(), "No graphics card. In the right panel choose Session options > Accelerator > GPU T4 x2, then run again."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")

In [ ]:
import glob, os, shutil
files = sorted(glob.glob("/kaggle/input/**/p1_*.npz", recursive=True))
assert files, "Patient 1's X-ray was not found. Click 'Add Input' and add the dataset patient1-xray."
os.makedirs("/tmp/p1", exist_ok=True)
for f in files:
    shutil.copy(f, "/tmp/p1/")
NAMES = " ".join(os.path.basename(f)[:-4] for f in files)
print("X-ray versions:", NAMES)

models = sorted(glob.glob("/kaggle/input/**/runs/full/best.pt", recursive=True))
assert models, "The trained model was not found. Click 'Add Input' and add the output of the notebook xray-to-3d-training."
MODEL = models[-1]
print("model file:", MODEL, round(os.path.getsize(MODEL) / 1e6), "MB")

In [ ]:
os.makedirs("/kaggle/working/project/src", exist_ok=True)
os.makedirs("/kaggle/working/project/scripts", exist_ok=True)

The next 3 cells write the project's code files into Kaggle. They are copies of the files in the project folder.

In [ ]:
%%writefile /kaggle/working/project/src/model.py
"""The model: gets a panoramic X-ray, outputs the straightened 3D block.

Plain description:
1. A "U-Net" looks at the X-ray. A U-Net is a common program for pictures. It shrinks the picture
   step by step to find large shapes (a whole jaw), then grows it back step by step to find small
   shapes (one tooth root), and at each size it reuses what it found on the way down.
2. For each X-ray pixel the U-Net outputs a list of numbers. The list is turned into a short line
   of points going from the tongue side to the cheek side. All lines together are the 3D block.
3. A small 3D step looks at each point together with its neighbours in all three directions and
   outputs, for each point: how bright it is in a 3D scan, and which part it is (bone, tooth, ...).
"""
import torch
import torch.nn as nn
import torch.nn.functional as F


def conv_block(n_in, n_out):
    return nn.Sequential(
        nn.Conv2d(n_in, n_out, 3, padding=1), nn.GroupNorm(8, n_out), nn.ReLU(inplace=True),
        nn.Conv2d(n_out, n_out, 3, padding=1), nn.GroupNorm(8, n_out), nn.ReLU(inplace=True))


class XrayTo3D(nn.Module):
    def __init__(self, in_channels=1, depth=64, n_classes=6, widths=(48, 96, 192, 384, 512), lift=8):
        """in_channels: 1 for the X-ray alone. More when extra inputs (from the mouth video) are added.
        depth: number of points on each tongue-to-cheek line."""
        super().__init__()
        self.depth, self.n_classes, self.lift = depth, n_classes, lift
        self.down = nn.ModuleList()
        n = in_channels
        for w in widths:
            self.down.append(conv_block(n, w))
            n = w
        self.up = nn.ModuleList()
        for w in reversed(widths[:-1]):
            self.up.append(conv_block(n + w, w))
            n = w
        self.to_lines = nn.Conv2d(n, lift * depth, 1)
        self.head3d = nn.Sequential(
            nn.Conv3d(lift, 24, 3, padding=1), nn.GroupNorm(8, 24), nn.ReLU(inplace=True),
            nn.Conv3d(24, 24, 3, padding=1), nn.GroupNorm(8, 24), nn.ReLU(inplace=True),
            nn.Conv3d(24, 1 + n_classes, 1))

    def forward(self, xray):
        """xray: [batch, in_channels, rows, columns], rows and columns must be multiples of 16.
        Returns (brightness [batch, depth, rows, columns] between 0 and 1,
                 part scores [batch, n_classes, depth, rows, columns])."""
        kept = []
        x = xray
        for i, block in enumerate(self.down):
            x = block(x)
            if i < len(self.down) - 1:
                kept.append(x)
                x = F.max_pool2d(x, 2)
        for block in self.up:
            x = F.interpolate(x, scale_factor=2, mode="bilinear", align_corners=False)
            x = block(torch.cat([x, kept.pop()], dim=1))
        b, _, h, w = x.shape
        x = self.to_lines(x).view(b, self.lift, self.depth, h, w)
        x = self.head3d(x)
        return torch.sigmoid(x[:, 0]), x[:, 1:]


In [ ]:
%%writefile /kaggle/working/project/src/train_data.py
"""Load the prepared examples and cut random training pieces from them."""
import json
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F

OUTSIDE = 255
HU_LOW, HU_HIGH = -1000.0, 3000.0


def hu_to_unit(hu):
    """Hounsfield units -> numbers between 0 and 1 (what the model outputs)."""
    return (hu - HU_LOW) / (HU_HIGH - HU_LOW)


def unit_to_hu(x):
    return x * (HU_HIGH - HU_LOW) + HU_LOW


def make_split(folder, n_check=5, n_test=10, seed=0):
    """Split the scans into train / check / test. Saved in split.json so it never changes.
    "check" scans are used during training to see if the model is getting better.
    "test" scans are never used during training."""
    folder = Path(folder)
    path = folder / "split.json"
    if path.exists():
        return json.loads(path.read_text())
    nums = sorted(p.stem for p in folder.glob("*.npz"))
    random.Random(seed).shuffle(nums)
    split = dict(test=sorted(nums[:n_test]), check=sorted(nums[n_test:n_test + n_check]),
                 train=sorted(nums[n_test + n_check:]))
    path.write_text(json.dumps(split, indent=1))
    return split


class Examples:
    """Holds a list of prepared scans in memory.
    xray: [n, rows, columns] 0..1,  hu: [n, depth, rows, columns] Hounsfield units,
    labels: [n, depth, rows, columns] part numbers, 255 = outside the scan."""

    def __init__(self, folder, numbers):
        xs, hs, ls = [], [], []
        for num in numbers:
            with np.load(Path(folder) / f"{num}.npz") as f:
                xs.append(torch.from_numpy(f["xray"].astype(np.float32)))
                hs.append(torch.from_numpy(f["hu"]).permute(2, 0, 1).contiguous())
                ls.append(torch.from_numpy(f["labels"]).permute(2, 0, 1).contiguous())
        self.numbers = list(numbers)
        self.xray, self.hu, self.labels = torch.stack(xs), torch.stack(hs), torch.stack(ls)

    def __len__(self):
        return len(self.numbers)

    def whole(self, i):
        """One whole example, as a batch of 1."""
        return self.xray[i:i + 1, None], self.hu[i:i + 1], self.labels[i:i + 1]

    def random_batch(self, batch_size, piece_cols, max_shift_rows=8):
        """Random pieces for training. Each piece is piece_cols columns wide and is randomly
        mirrored left-right and moved up or down a little, so the model sees more variety."""
        rows, cols = self.xray.shape[1:]
        xs, hs, ls = [], [], []
        for _ in range(batch_size):
            i = random.randrange(len(self))
            c = random.randrange(0, cols - piece_cols + 1)
            x, h, l = self.xray[i, :, c:c + piece_cols], self.hu[i, :, :, c:c + piece_cols], self.labels[i, :, :, c:c + piece_cols]
            if random.random() < 0.5:
                x, h, l = x.flip(-1), h.flip(-1), l.flip(-1)
            s = random.randint(-max_shift_rows, max_shift_rows)
            if s != 0:
                x, h, l = _shift_rows(x, s, 0.0), _shift_rows(h, s, int(HU_LOW)), _shift_rows(l, s, OUTSIDE)
            xs.append(x); hs.append(h); ls.append(l)
        return torch.stack(xs)[:, None], torch.stack(hs), torch.stack(ls)


def _shift_rows(t, s, fill):
    """Move the picture s rows down (or up if s is negative). Rows that become empty get `fill`."""
    out = torch.full_like(t, fill)
    if s > 0:
        out[..., s:, :] = t[..., :-s, :]
    else:
        out[..., :s, :] = t[..., -s:, :]
    return out


def change_xray_look(xray):
    """Randomly change brightness curve, contrast, blur and noise of a batch of X-rays [b, 1, rows, columns].
    Real X-rays do not look exactly like our fake ones, so the model must not depend on the exact look."""
    b = xray.shape[0]
    dev = xray.device
    gamma = torch.empty(b, 1, 1, 1, device=dev).uniform_(0.7, 1.5)
    x = xray.clamp(0, 1) ** gamma
    contrast = torch.empty(b, 1, 1, 1, device=dev).uniform_(0.75, 1.25)
    bright = torch.empty(b, 1, 1, 1, device=dev).uniform_(-0.08, 0.08)
    x = (x - 0.5) * contrast + 0.5 + bright
    if random.random() < 0.5:
        sigma = random.uniform(0.5, 2.0)
        k = torch.arange(-6, 7, device=dev, dtype=x.dtype)
        k = torch.exp(-0.5 * (k / sigma) ** 2)
        k = k / k.sum()
        x = F.conv2d(F.pad(x, (6, 6, 0, 0), mode="replicate"), k.view(1, 1, 1, -1))
        x = F.conv2d(F.pad(x, (0, 0, 6, 6), mode="replicate"), k.view(1, 1, -1, 1))
    x = x + torch.randn_like(x) * random.uniform(0.0, 0.04)
    return x.clamp(0, 1)


In [ ]:
%%writefile /kaggle/working/project/scripts/predict.py
"""Run the trained model on prepared scans and save what it outputs.

Run:  python scripts/predict.py --data PREPARED_FOLDER --model best.pt --out predictions/test --group test

For each scan this gives the model the fake X-ray of that scan and saves the model's output in one
file, SCAN_NUMBER.npz, with:
  hu      the brightness of every point of the block, in Hounsfield units, [rows, columns, depth]
  labels  the part number of every point of the block (0 background, 1 lower jaw, 2 upper jaw,
          3 nerve canal, 4 sinus, 5 teeth), [rows, columns, depth]
This is the same layout as the prepared scan files, so the two can be compared point by point.
The comparing is done by scripts/measure.py.

--group says which scans to use:
  test   the 10 scans that were never used during training (the default)
  check  the 5 scans that were used during training to pick the best saved model
  train  the 46 scans the model was trained on
"""
import argparse
import json
import sys
import time
from pathlib import Path

import numpy as np
import torch

ROOT = Path(__file__).resolve().parents[1]
sys.path.insert(0, str(ROOT))
from src.model import XrayTo3D
from src.train_data import make_split, unit_to_hu


def load_model(path, device):
    """Load a model file saved by scripts/train.py."""
    saved = torch.load(path, map_location=device, weights_only=False)  # our own file, so it is safe to load fully
    widths = (16, 32, 48, 64, 64) if saved.get("small") else (48, 96, 192, 384, 512)
    model = XrayTo3D(depth=saved["depth"], widths=widths).to(device)
    model.load_state_dict(saved["model"])
    model.eval()
    return model, saved


@torch.no_grad()
def predict_one(model, xray, device):
    """xray: [rows, columns] numbers from 0 to 1. Returns (hu, labels), both [rows, columns, depth]."""
    x = torch.from_numpy(xray.astype(np.float32))[None, None].to(device)
    with torch.autocast(device.type, enabled=device.type == "cuda"):
        bright, scores = model(x)
    hu = unit_to_hu(bright[0].float()).round().clamp(-1000, 3000).to(torch.int16)
    labels = scores[0].argmax(dim=0).to(torch.uint8)
    return hu.permute(1, 2, 0).cpu().numpy(), labels.permute(1, 2, 0).cpu().numpy()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data", required=True, help="folder made by prepare_tf3.py")
    ap.add_argument("--model", required=True, help="model file saved by train.py, for example best.pt")
    ap.add_argument("--out", required=True, help="folder for the model's outputs")
    ap.add_argument("--group", default="test", choices=["test", "check", "train"])
    ap.add_argument("--scans", nargs="*", help="scan numbers to use instead of a whole group")
    args = ap.parse_args()

    out = Path(args.out); out.mkdir(parents=True, exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"running on: {device}" + (f" ({torch.cuda.get_device_name(0)})" if device.type == "cuda" else ""), flush=True)

    numbers = args.scans if args.scans else make_split(args.data)[args.group]

    # The training script saved the list of scans it trained on next to the model file.
    # Stop if a scan we are about to score as "never used" was in fact used.
    settings_path = Path(args.model).parent / "settings.json"
    if settings_path.exists():
        settings = json.loads(settings_path.read_text())
        used = set(settings["train"]) | (set(settings["check"]) if args.group == "test" else set())
        if args.group != "train":
            overlap = sorted(set(numbers) & used)
            if overlap:
                raise SystemExit(f"ERROR: these scans were used during training: {overlap}")
            print(f"checked against {settings_path.name}: none of the {len(numbers)} scans was used for training", flush=True)
    else:
        print("no settings.json next to the model file, so the scan list could not be checked", flush=True)

    model, saved = load_model(args.model, device)
    print(f"loaded {args.model} (saved after epoch {saved['epoch']})", flush=True)
    for i, num in enumerate(numbers):
        t0 = time.time()
        with np.load(Path(args.data) / f"{num}.npz") as f:
            xray = f["xray"]
        hu, labels = predict_one(model, xray, device)
        np.savez_compressed(out / f"{num}.npz", hu=hu, labels=labels)
        print(f"[{i + 1}/{len(numbers)}] scan {num} done in {time.time() - t0:.1f}s", flush=True)
    (out / "about.json").write_text(json.dumps(dict(model=str(args.model), epoch=saved["epoch"], group="given list" if args.scans else args.group, scans=list(numbers)), indent=1))
    print(f"{len(numbers)} outputs are in {out}")


if __name__ == "__main__":
    main()


The next cell runs the model on each version of Patient 1's X-ray.

In [ ]:
!python /kaggle/working/project/scripts/predict.py --data /tmp/p1 --model "$MODEL" --out /kaggle/working/predictions/patient1 --scans $NAMES